In [ ]:
import os, re
import numpy as np
import pandas as pd
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
GPDS_ROOT = r"W:\SRH study\Case Study 2\Offline Signature Verification\Datasets\GPDS150"
print("Exists:", os.path.exists(GPDS_ROOT))

In [ ]:
def parse_gpds150(gpds_root):
    rows = []
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

    for split in ["train", "test"]:
        split_dir = os.path.join(gpds_root, split)
        if not os.path.exists(split_dir):
            continue

        for writer in sorted(os.listdir(split_dir)):
            writer_dir = os.path.join(split_dir, writer)
            if not os.path.isdir(writer_dir):
                continue

            for label_folder in ["genuine", "forged", "forgery"]:
                lf_dir = os.path.join(writer_dir, label_folder)
                if not os.path.exists(lf_dir):
                    continue

                for fname in os.listdir(lf_dir):
                    ext = Path(fname).suffix.lower()
                    if ext not in exts:
                        continue

                    path = os.path.join(lf_dir, fname)

                    # Extract writer_id and sample_id from filename like c-150-18 (Copy).jpg
                    # Pattern: c-<writer>-<sample>
                    m = re.search(r"c-(\d+)-(\d+)", fname)
                    writer_id = int(m.group(1)) if m else int(writer)
                    sample_id = int(m.group(2)) if m else None

                    label = "genuine" if label_folder == "genuine" else "forgery"

                    rows.append({
                        "dataset": "GPDS150",
                        "split": split,
                        "writer_id": writer_id,
                        "sample_id": sample_id,
                        "label": label,
                        "path": path,
                        "filename": fname
                    })

    df = pd.DataFrame(rows)
    return df

valid_gpds = parse_gpds150(GPDS_ROOT)

print("Total images:", len(valid_gpds))
print(valid_gpds.groupby(["split","label"]).size())
print("Unique writers:", valid_gpds["writer_id"].nunique())
display(valid_gpds.head())

In [ ]:
print("Missing sample_id:", valid_gpds["sample_id"].isna().sum())

# Optional: check duplicate paths / filenames
print("Duplicate paths:", valid_gpds["path"].duplicated().sum())
print("Duplicate filenames within same writer/label/split:",
      valid_gpds.duplicated(subset=["split","writer_id","label","filename"]).sum())

In [ ]:
train_df = valid_gpds[valid_gpds["split"] == "train"].copy()
test_df  = valid_gpds[valid_gpds["split"] == "test"].copy()

train_writers_all = np.array(sorted(train_df["writer_id"].unique()))
rng = np.random.default_rng(42)
rng.shuffle(train_writers_all)

val_ratio = 0.2
n_val = int(round(len(train_writers_all) * val_ratio))

val_writers = set(train_writers_all[:n_val])
train_writers = set(train_writers_all[n_val:])
test_writers = set(test_df["writer_id"].unique())

print("Train writers:", len(train_writers))
print("Val writers:", len(val_writers))
print("Test writers:", len(test_writers))
print("Overlap train-val:", len(train_writers & val_writers))
print("Overlap with test (should be 0 ideally):", len((train_writers | val_writers) & test_writers))

In [ ]:
def build_pools(df_subset):
    genuine_by_writer = {}
    forgery_by_writer = {}
    for wid, g in df_subset.groupby("writer_id"):
        gg = g[g["label"] == "genuine"]["path"].tolist()
        ff = g[g["label"] == "forgery"]["path"].tolist()
        if len(gg) > 1:
            genuine_by_writer[wid] = gg
        if len(ff) > 0:
            forgery_by_writer[wid] = ff
    return genuine_by_writer, forgery_by_writer

def generate_pairs(df, writer_set, n_pairs=80000, seed=1, neg_mix=0.9):
    df_subset = df[df["writer_id"].isin(writer_set)].copy()
    genuine_by_writer, forgery_by_writer = build_pools(df_subset)

    writers = sorted(genuine_by_writer.keys())
    writers_with_forg = sorted(set(genuine_by_writer) & set(forgery_by_writer))
    if len(writers) < 2:
        raise ValueError("Need >= 2 writers with >=2 genuine samples each.")
    if len(writers_with_forg) == 0:
        raise ValueError("Need writers with forgeries too.")

    rng = np.random.default_rng(seed)
    n_pos = n_pairs // 2
    n_neg = n_pairs - n_pos
    n_neg_same  = int(round(n_neg * neg_mix))
    n_neg_cross = n_neg - n_neg_same

    pairs = []

    # Positive: genuine-genuine same writer
    for _ in range(n_pos):
        w = rng.choice(writers)
        g = genuine_by_writer[w]
        i, j = rng.choice(len(g), size=2, replace=False)
        pairs.append({"path_a": g[i], "path_b": g[j], "label": 1})

    # Negative: genuine-forgery same writer
    for _ in range(n_neg_same):
        w = rng.choice(writers_with_forg)
        g = genuine_by_writer[w]
        f = forgery_by_writer[w]
        i = rng.integers(0, len(g))
        j = rng.integers(0, len(f))
        pairs.append({"path_a": g[i], "path_b": f[j], "label": 0})

    # Negative: genuine-genuine different writers
    for _ in range(n_neg_cross):
        w1, w2 = rng.choice(writers, size=2, replace=False)
        g1 = genuine_by_writer[w1]
        g2 = genuine_by_writer[w2]
        i = rng.integers(0, len(g1))
        j = rng.integers(0, len(g2))
        pairs.append({"path_a": g1[i], "path_b": g2[j], "label": 0})

    return pd.DataFrame(pairs).sample(frac=1.0, random_state=seed).reset_index(drop=True)

train_pairs = generate_pairs(train_df, train_writers, n_pairs=80000, seed=1, neg_mix=0.9)
val_pairs   = generate_pairs(train_df, val_writers,   n_pairs=20000, seed=2, neg_mix=0.9)
test_pairs  = generate_pairs(test_df,  test_writers,  n_pairs=20000, seed=3, neg_mix=0.9)

print("Train pairs:", len(train_pairs), "Val pairs:", len(val_pairs), "Test pairs:", len(test_pairs))
print("Train balance:\n", train_pairs["label"].value_counts(normalize=True))

In [ ]:
IMG_SIZE = (224, 224)

def load_preprocess(path):
    path = path.numpy().decode("utf-8")
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Failed to read: {path}")

    img = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_AREA)
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=-1)
    return img

def tf_load_preprocess(path):
    img = tf.py_function(load_preprocess, [path], Tout=tf.float32)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 1])
    return img

In [ ]:
def make_pair_dataset(pairs_df, batch_size=32, shuffle=True):
    a = pairs_df["path_a"].astype(str).values
    b = pairs_df["path_b"].astype(str).values
    y = pairs_df["label"].astype(np.float32).values

    ds = tf.data.Dataset.from_tensor_slices((a, b, y))

    def map_fn(pa, pb, lab):
        ia = tf_load_preprocess(pa)
        ib = tf_load_preprocess(pb)
        return (ia, ib), lab

    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(4000, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_pair_dataset(train_pairs, batch_size=32, shuffle=True)
val_ds   = make_pair_dataset(val_pairs,   batch_size=32, shuffle=False)
test_ds  = make_pair_dataset(test_pairs,  batch_size=32, shuffle=False)

(xb, yb) = next(iter(train_ds))
print(xb[0].shape, xb[1].shape, yb.shape)
print("Batch label counts:", np.unique(yb.numpy(), return_counts=True))